In [ ]:
# MPAGE Package Testing
# This notebook demonstrates basic functionality of the MPAGE package

# Load the package
devtools::load_all()

# Build PPI network using STRING database
net <- build_ppi_network(
    proteins = NULL,  # Use full network
    data_sources = c("STRING"),
    min_confidence = 0.7,
    species = "Homo sapiens",
    string_version = "12"
)

# Display network information
print(net)

# Summary statistics
cat("Network Summary:\n")
cat("Nodes:", length(V(net)), "\n")
cat("Edges:", length(E(net)), "\n")
cat("Average degree:", mean(degree(net)), "\n")

In [ ]:
net <- build_ppi_network(
    proteins = NULL,
    data_sources = c("STRING"),
    min_confidence = 0.7,
    include_experimental = TRUE,
    exclude_predicted = FALSE,
    species = "Homo sapiens"
)


In [ ]:
net


IGRAPH f55e4a4 UNW- 16201 473860 -- 
+ attr: name (v/c), weight (e/n), source (e/c)
+ edges from f55e4a4 (vertex names):
 [1] ARF5 --ACAP1     ARF5 --COPA      ARF5 --RAB11FIP3 ARF5 --COPB2    
 [5] ARF5 --COPE      ARF5 --ARF3      ARF5 --ARFGAP1   ARF5 --RAB11FIP1
 [9] ARF5 --CYTH1     ARF5 --IQSEC1    ARF5 --RAB11A    ARF5 --ARFGAP2  
[13] ARF5 --COPB1     ARF5 --ARF4      ARF5 --ASAP1     ARF5 --RAB11FIP4
[17] ARF5 --GORAB     ARF5 --ACAP2     ARF5 --ARFGAP3   ARF5 --ARFIP1   
[21] ARF5 --ARF1      ARF5 --CDKN2A    ARF5 --GBF1      ARF5 --ARFIP2   
[25] ARF5 --AGAP1     ARF5 --ASAP2     ARF5 --COPZ1     M6PR --GGA3     
[29] M6PR --VTI1A     M6PR --TGOLN2    M6PR --GGA2      M6PR --GGA1     
+ ... omitted several edges

In [3]:
string_db <- STRINGdb::STRINGdb$new(
    species = 9606,
    version = "12",
    score_threshold = 0.7 * 1000
)


In [16]:
all_interactions <- string_db$get_interactions(
    string_db$get_proteins()$protein_external_id
)


In [1]:
library(dplyr)
library(stringr)
library(stringi)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
# IntAct Database Preprocessing
# This cell demonstrates processing IntAct PPI data

# Load required libraries
library(dplyr)
library(stringr)
library(data.table)
library(igraph)

# IntAct preprocessing function
preprocess_intact_data <- function(input_file, output_file = NULL, species_filter = c("9606", "10090")) {
    """
    Preprocess IntAct database file for MPAGE package
    
    Args:
        input_file: Path to IntAct TSV file
        output_file: Path to save processed RDS file (optional)
        species_filter: Vector of NCBI taxonomy IDs to filter
    
    Returns:
        Cleaned interaction data frame
    """
    
    message("Loading IntAct data from: ", input_file)
    
    # Read IntAct data
    df <- data.table::fread(input_file, sep = "\t", check.names = TRUE)
    
    message("Processing ", nrow(df), " interactions...")
    
    # Select relevant columns and extract gene names
    processed_data <- df %>%
        dplyr::select(
            Alt..ID.s..interactor.A,
            Alt..ID.s..interactor.B,
            Alias.es..interactor.A,
            Alias.es..interactor.B,
            Taxid.interactor.A,
            Taxid.interactor.B,
            Confidence.value.s.
        ) %>%
        # Extract species information
        dplyr::mutate(
            taxidA = str_extract(Taxid.interactor.A, "taxid:([0-9]+)") %>% str_remove("taxid:"),
            taxidB = str_extract(Taxid.interactor.B, "taxid:([0-9]+)") %>% str_remove("taxid:")
        ) %>%
        # Filter by species
        dplyr::filter(
            taxidA %in% species_filter &
            taxidB %in% species_filter &
            taxidA == taxidB
        ) %>%
        # Extract gene names from aliases
        dplyr::mutate(
            geneA = str_extract(Alias.es..interactor.A, "uniprotkb:[A-Za-z0-9-]+\(gene name\)") %>% 
                   str_remove("uniprotkb:") %>% str_remove("\(gene name\)") %>%
                   str_to_upper(),
            geneB = str_extract(Alias.es..interactor.B, "uniprotkb:[A-Za-z0-9-]+\(gene name\)") %>% 
                   str_remove("uniprotkb:") %>% str_remove("\(gene name\)") %>%
                   str_to_upper(),
            score = str_extract(Confidence.value.s., "intact-miscore:([0-9\\.]+)") %>% 
                   str_remove("intact-miscore:") %>% 
                   as.numeric(),
            taxid = taxidA
        ) %>%
        # Clean and filter
        dplyr::select(geneA, geneB, score, taxid) %>%
        dplyr::filter(!is.na(geneA), !is.na(geneB), !is.na(score)) %>%
        dplyr::distinct() %>%
        dplyr::arrange(desc(score))
    
    # Create summary statistics
    cat("Processing complete:\n")
    cat("Total interactions:", nrow(processed_data), "\n")
    cat("Unique proteins:", length(unique(c(processed_data$geneA, processed_data$geneB))), "\n")
    cat("Species:", paste(unique(processed_data$taxid), collapse = ", "), "\n")
    cat("Score range:", range(processed_data$score, na.rm = TRUE), "\n")
    
    # Save if output file provided
    if (!is.null(output_file)) {
        saveRDS(processed_data, output_file)
        message("Saved processed data to: ", output_file)
    }
    
    return(processed_data)
}

# Test the preprocessing function
input_file <- "/workspaces/MPAGE/data-raw/intact.txt"
output_file <- "/workspaces/MPAGE/data-raw/intact_processed.rds"

if (file.exists(input_file)) {
    intact_data <- preprocess_intact_data(input_file, output_file)
    
    # Display summary
    cat("\n=== IntAct Data Summary ===\n")
    print(intact_data %>% head())
    
    # Species breakdown
    cat("\n=== Species Breakdown ===\n")
    intact_data %>%
        group_by(taxid) %>%
        summarise(interactions = n(), .groups = 'drop') %>%
        print()
    
} else {
    cat("Input file not found: ", input_file)
}

In [ ]:
# MPAGE Package Testing and Validation
# This notebook demonstrates core functionality and validates the package

# Load required libraries
library(dplyr)
library(igraph)
library(ggplot2)

# Load the package
devtools::load_all()

# Test 1: Basic PPI network construction
message("=== Testing PPI Network Construction ===")

# Build full STRING network
net_full <- build_ppi_network(
    proteins = NULL,
    data_sources = c("STRING"),
    min_confidence = 0.7,
    species = "Homo sapiens",
    string_version = "12"
)

# Network summary
cat("Full STRING Network:\n")
cat(sprintf("Nodes: %d\n", length(V(net_full))))
cat(sprintf("Edges: %d\n", length(E(net_full))))
cat(sprintf("Average degree: %.2f\n", mean(degree(net_full))))

# Test 2: RNA modification proteins
message("\n=== Testing RNA Modification Proteins ===")

# Get all m6A proteins
m6a_proteins <- get_rna_mod_proteins(
    modification_types = "m6A",
    use_built_in = TRUE
)

cat("m6A proteins found:", nrow(m6a_proteins), "\n")
print(head(m6a_proteins))

# Test 3: Subnetwork extraction
message("\n=== Testing Subnetwork Extraction ===")

# Extract subnetwork for m6A proteins
m6a_network <- build_ppi_network(
    proteins = unique(m6a_proteins$gene_symbol),
    data_sources = c("STRING"),
    min_confidence = 0.7,
    species = "Homo sapiens"
)

cat("m6A subnetwork:\n")
cat(sprintf("Nodes: %d\n", length(V(m6a_network))))
cat(sprintf("Edges: %d\n", length(E(m6a_network))))

# Test 4: IntAct data processing (if available)
if (file.exists("data-raw/intact.txt")) {
    message("\n=== Testing IntAct Processing ===")
    
    # Process IntAct data
    intact_processed <- preprocess_intact_data(
        input_file = "data-raw/intact.txt",
        species_filter = c("9606", "10090"),
        min_score = 0.3
    )
    
    # Create IntAct network
    intact_net <- load_intact_network(
        processed_file = "data-raw/intact_processed.rds",
        species = "9606",
        min_score = 0.3
    )
    
    cat("IntAct Network:\n")
    cat(sprintf("Nodes: %d\n", length(V(intact_net))))
    cat(sprintf("Edges: %d\n", length(E(intact_net))))
} else {
    message("\n=== Skipping IntAct Tests (data file not found) ===")
}

# Test 5: Module identification
message("\n=== Testing Module Identification ===")

# Identify modules in m6A network
if (length(V(m6a_network)) > 0) {
    modules <- identify_modules(
        ppi_network = m6a_network,
        algorithm = "FASTGREEDY"
    )
    
    cat("Modules found:", length(modules), "\n")
    cat("Module sizes:", sapply(modules, length), "\n")
} else {
    cat("No m6A network found for module identification\n")
}

# Test 6: Performance summary
message("\n=== Performance Summary ===")

# Create summary data frame
summary_df <- data.frame(
    Network = c("Full STRING", "m6A STRING", "IntAct"),
    Nodes = c(length(V(net_full)), length(V(m6a_network)), 
              ifelse(exists("intact_net"), length(V(intact_net)), NA)),
    Edges = c(length(E(net_full)), length(E(m6a_network)), 
              ifelse(exists("intact_net"), length(E(intact_net)), NA)),
    stringsAsFactors = FALSE
)

print(summary_df)

# Test 7: Validation checks
message("\n=== Validation Checks ===")

# Check for missing values
validation_results <- list(
    full_network_valid = !is.null(net_full) && length(V(net_full)) > 0,
    m6a_proteins_valid = nrow(m6a_proteins) > 0,
    m6a_network_valid = !is.null(m6a_network) && length(V(m6a_network)) > 0
)

print(validation_results)

# Save test results
saveRDS(list(
    full_network = net_full,
    m6a_proteins = m6a_proteins,
    m6a_network = m6a_network,
    summary = summary_df,
    validation = validation_results
), "testing_results.rds")

message("\n=== Testing Complete ===")
message("Results saved to testing_results.rds")

In [ ]:
saveRDS(all_interactions, "/workspaces/MPAGE/data-raw/intact_igraph.rds")


In [ ]:
all_interactions %>% head()


geneA,geneB,score,taxid
<chr>,<chr>,<dbl>,<chr>
SYNJ1,AMPH,0.67,9606
SYNJ1,SH3GL1,0.44,9606
DNM1,AMPH,0.82,9606
DNM1,SH3GL1,0.44,9606
DNM2,SH3GL1,0.35,9606
DNM2,AMPH,0.71,9606


In [ ]:
all_interactions %>%
    group_by(taxid) %>%
    summarise(n = n())


taxid,n
<chr>,<int>
10090,21458
9606,595011


In [ ]:
saveRDS(net, "/workspaces/MPAGE/data-raw/intact_igraph.rds")


In [47]:
df %>% dplyr::filter(Confidence.value.s. == "intact-miscore:0.53", str_detect(Alias.es..interactor.B, "PCBP2"))


X.ID.s..interactor.A,ID.s..interactor.B,Alt..ID.s..interactor.A,Alt..ID.s..interactor.B,Alias.es..interactor.A,Alias.es..interactor.B,Interaction.detection.method.s.,Publication.1st.author.s.,Publication.Identifier.s.,Taxid.interactor.A,⋯,Checksum.s..interactor.A,Checksum.s..interactor.B,Interaction.Checksum.s.,Negative,Feature.s..interactor.A,Feature.s..interactor.B,Stoichiometry.s..interactor.A,Stoichiometry.s..interactor.B,Identification.method.participant.A,Identification.method.participant.B
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
uniprotkb:P11940,uniprotkb:Q15366,intact:EBI-81531|intact:EBI-28983164|uniprotkb:Q15097|uniprotkb:Q93004|ensembl:ENSP00000313007.5|ensembl:ENSP00000428749.2|ensembl:ENSP00000428840.2|ensembl:ENSP00000429119.2|ensembl:ENSP00000429790.2|ensembl:ENSP00000429892.2|ensembl:ENSP00000430012.2,intact:EBI-945799|uniprotkb:Q6PKG5|intact:EBI-9640573|uniprotkb:Q6IPF4|intact:EBI-2555278|uniprotkb:A8K7X6|uniprotkb:F8VYL7|uniprotkb:Q32Q82|uniprotkb:Q68Y55|uniprotkb:I6L8F9|uniprotkb:Q59HD4|uniprotkb:G3V0E8|uniprotkb:B4DXP5|ensembl:ENSP00000408949.2,psi-mi:pabp1_human(display_long)|uniprotkb:PABPC1(gene name)|psi-mi:PABPC1(display_short)|uniprotkb:PAB1(gene name synonym)|uniprotkb:PABP1(gene name synonym)|uniprotkb:PABPC2(gene name synonym)|uniprotkb:PABP(gene name synonym),psi-mi:pcbp2_human(display_long)|uniprotkb:PCBP2(gene name)|psi-mi:PCBP2(display_short)|uniprotkb:Alpha-CP2(gene name synonym)|uniprotkb:Heterogeneous nuclear ribonucleoprotein E2(gene name synonym),"psi-mi:""MI:0096""(pull down)",Wang et al. (1999),pubmed:10373504|mint:MINT-5211576,taxid:9606(human)|taxid:9606(Homo sapiens),⋯,rogid:XudTSPq2h9ckKAc48X88RVafPRU9606,rogid:KWabGhH+pfSX5k0vzs3SXH8Agaw9606,intact-crc:7139D82B3D772339|rigid:j5oioWbwKbmgKYXOplV2QD/1WdA,FALSE,-,-,-,-,"psi-mi:""MI:0396""(predetermined participant)","psi-mi:""MI:0396""(predetermined participant)"
uniprotkb:Q12771,uniprotkb:Q15366,intact:EBI-8609859|intact:MINT-1526261,intact:EBI-945799|uniprotkb:Q6PKG5|intact:EBI-9640573|uniprotkb:Q6IPF4|intact:EBI-2555278|uniprotkb:A8K7X6|uniprotkb:F8VYL7|uniprotkb:Q32Q82|uniprotkb:Q68Y55|uniprotkb:I6L8F9|uniprotkb:Q59HD4|uniprotkb:G3V0E8|uniprotkb:B4DXP5|ensembl:ENSP00000408949.2,psi-mi:q12771_human(display_long),psi-mi:pcbp2_human(display_long)|uniprotkb:PCBP2(gene name)|psi-mi:PCBP2(display_short)|uniprotkb:Alpha-CP2(gene name synonym)|uniprotkb:Heterogeneous nuclear ribonucleoprotein E2(gene name synonym),"psi-mi:""MI:0018""(two hybrid)",Kiledjian et al. (1997),mint:MINT-5221578|pubmed:9234743,taxid:9606(human)|taxid:9606(Homo sapiens),⋯,rogid:wNHcwFLt1g3q5j3DCQ/SW6u05VM9606,rogid:KWabGhH+pfSX5k0vzs3SXH8Agaw9606,intact-crc:AA7DB477AC2E408B|rigid:7H7szteIW4uF3ThIU5XTXwlvE6w,FALSE,tag:?-?(MINT-1526273),tag:?-?(MINT-1526279),-,-,"psi-mi:""MI:0078""(nucleotide sequence identification)","psi-mi:""MI:0078""(nucleotide sequence identification)"
uniprotkb:Q12771,uniprotkb:Q15366,intact:EBI-8609859|intact:MINT-1526261,intact:EBI-945799|uniprotkb:Q6PKG5|intact:EBI-9640573|uniprotkb:Q6IPF4|intact:EBI-2555278|uniprotkb:A8K7X6|uniprotkb:F8VYL7|uniprotkb:Q32Q82|uniprotkb:Q68Y55|uniprotkb:I6L8F9|uniprotkb:Q59HD4|uniprotkb:G3V0E8|uniprotkb:B4DXP5|ensembl:ENSP00000408949.2,psi-mi:q12771_human(display_long),psi-mi:pcbp2_human(display_long)|uniprotkb:PCBP2(gene name)|psi-mi:PCBP2(display_short)|uniprotkb:Alpha-CP2(gene name synonym)|uniprotkb:Heterogeneous nuclear ribonucleoprotein E2(gene name synonym),"psi-mi:""MI:0096""(pull down)",Kiledjian et al. (1997),mint:MINT-5221578|pubmed:9234743,taxid:9606(human)|taxid:9606(Homo sapiens),⋯,rogid:wNHcwFLt1g3q5j3DCQ/SW6u05VM9606,rogid:KWabGhH+pfSX5k0vzs3SXH8Agaw9606,intact-crc:C93CECC87A14F52E|rigid:7H7szteIW4uF3ThIU5XTXwlvE6w,FALSE,-,-,-,-,"psi-mi:""MI:0396""(predetermined participant)","psi-mi:""MI:0396""(predetermined participant)"


In [ ]:
all_interactions %>%
    dplyr::filter(is.na(geneA))


geneA,geneB,score
<chr>,<chr>,<dbl>
NA,PCBP2,0.53
NA,SYNJ1,0.56
NA,SNW1,0.44
NA,RAD51B,0.56
NA,BAX,0.59
NA,B2M,0.44
NA,B2M,0.88
NA,NA,0.86
NA,F8,0.44


In [ ]:
string_ids <- unique(c(all_interactions$from, all_interactions$to))
mapped_proteins <- string_db$map(string_ids, "gene", removeUnmapped = TRUE)


ERROR: Error in matrix(NA, length(strColsFrom), nrow(dfToMap)): non-numeric matrix extent


In [ ]:
string_ids


[1] "9606.ENSP00000000233" "9606.ENSP00000000412" "9606.ENSP00000001008"
    [4] "9606.ENSP00000001146" "9606.ENSP00000002125" "9606.ENSP00000002165"
    [7] "9606.ENSP00000002596" "9606.ENSP00000002829" "9606.ENSP00000003084"
   [10] "9606.ENSP00000003100" "9606.ENSP00000003302" "9606.ENSP00000004531"
   [13] "9606.ENSP00000004982" "9606.ENSP00000005178" "9606.ENSP00000005226"
   [16] "9606.ENSP00000005257" "9606.ENSP00000005260" "9606.ENSP00000005284"
   [19] "9606.ENSP00000005286" "9606.ENSP00000005340" "9606.ENSP00000005386"
   [22] "9606.ENSP00000005587" "9606.ENSP00000006015" "9606.ENSP00000006053"
   [25] "9606.ENSP00000006275" "9606.ENSP00000006526" "9606.ENSP00000006658"
   [28] "9606.ENSP00000007390" "9606.ENSP00000007414" "9606.ENSP00000007699"
   [31] "9606.ENSP00000007735" "9606.ENSP00000008391" "9606.ENSP00000008527"
   [34] "9606.ENSP00000008938" "9606.ENSP00000009041" "9606.ENSP00000009105"
   [37] "9606.ENSP00000009530" "9606.ENSP00000011292" "9606.ENSP00000011473"
   [40] "9606.ENSP00000011619" "9606.ENSP00000011653" "9606.ENSP00000011898"
   [43] "9606.ENSP00000012049" "9606.ENSP00000012443" "9606.ENSP00000013070"
   [46] "9606.ENSP00000013125" "9606.ENSP00000013222" "9606.ENSP00000013807"
   [49] "9606.ENSP00000014914" "9606.ENSP00000014930" "9606.ENSP00000016171"
   [52] "9606.ENSP00000016913" "9606.ENSP00000016946" "9606.ENSP00000017003"
   [55] "9606.ENSP00000019103" "9606.ENSP00000020926" "9606.ENSP00000020945"
   [58] "9606.ENSP00000023064" "9606.ENSP00000023939" "9606.ENSP00000025008"
   [61] "9606.ENSP00000025301" "9606.ENSP00000026218" "9606.ENSP00000027335"
   [64] "9606.ENSP00000029410" "9606.ENSP00000035307" "9606.ENSP00000037243"
   [67] "9606.ENSP00000037502" "9606.ENSP00000039007" "9606.ENSP00000040584"
   [70] "9606.ENSP00000040663" "9606.ENSP00000040738" "9606.ENSP00000040877"
   [73] "9606.ENSP00000043402" "9606.ENSP00000044462" "9606.ENSP00000045083"
   [76] "9606.ENSP00000046087" "9606.ENSP00000046794" "9606.ENSP00000052754"
   [79] "9606.ENSP00000053243" "9606.ENSP00000053468" "9606.ENSP00000053469"
   [82] "9606.ENSP00000053867" "9606.ENSP00000054650" "9606.ENSP00000054666"
   [85] "9606.ENSP00000054668" "9606.ENSP00000054950" "9606.ENSP00000055077"
   [88] "9606.ENSP00000055335" "9606.ENSP00000055682" "9606.ENSP00000056217"
   [91] "9606.ENSP00000056233" "9606.ENSP00000061240" "9606.ENSP00000064724"
   [94] "9606.ENSP00000064778" "9606.ENSP00000064780" "9606.ENSP00000070846"
   [97] "9606.ENSP00000072644" "9606.ENSP00000072869" "9606.ENSP00000074304"
  [100] "9606.ENSP00000075120" "9606.ENSP00000075503" "9606.ENSP00000078429"
  [103] "9606.ENSP00000078445" "9606.ENSP00000080059" "9606.ENSP00000081029"
  [106] "9606.ENSP00000083182" "9606.ENSP00000085068" "9606.ENSP00000085219"
  [109] "9606.ENSP00000086933" "9606.ENSP00000155093" "9606.ENSP00000155840"
  [112] "9606.ENSP00000155858" "9606.ENSP00000155926" "9606.ENSP00000156084"
  [115] "9606.ENSP00000156109" "9606.ENSP00000156471" "9606.ENSP00000156626"
  [118] "9606.ENSP00000157600" "9606.ENSP00000157812" "9606.ENSP00000158762"
  [121] "9606.ENSP00000158771" "9606.ENSP00000159060" "9606.ENSP00000159111"
  [124] "9606.ENSP00000160262" "9606.ENSP00000160373" "9606.ENSP00000160382"
  [127] "9606.ENSP00000160827" "9606.ENSP00000161559" "9606.ENSP00000161863"
  [130] "9606.ENSP00000162391" "9606.ENSP00000162749" "9606.ENSP00000163416"
  [133] "9606.ENSP00000164024" "9606.ENSP00000164133" "9606.ENSP00000164139"
  [136] "9606.ENSP00000164227" "9606.ENSP00000164305" "9606.ENSP00000165524"
  [139] "9606.ENSP00000166139" "9606.ENSP00000166244" "9606.ENSP00000166345"
  [142] "9606.ENSP00000167106" "9606.ENSP00000167586" "9606.ENSP00000167588"
  [145] "9606.ENSP00000168148" "9606.ENSP00000168216" "9606.ENSP00000168712"
  [148] "9606.ENSP00000169298" "9606.ENSP00000169551" "9606.ENSP00000170150"
  [151] "9606.ENSP00000170168" "9606.ENSP00000170447" "9606.ENSP00000170564"
  [154] "9606.ENSP00000171887" "9606.ENSP00000172229" "9606.ENSP00000173229"
